# ARM97 Pressure-Level Heatmap Plotting

This notebook creates time-pressure heatmaps for ARM97 variables with a vertical level dimension. Model hybrid levels are interpolated to ARM97 observation pressure levels before plotting.

Behavior:

- Variables with explicit observation mappings can show `observation`, `baseline`, and `baseline - observation` heatmaps.
- Variables without mapped observations show `baseline` heatmaps.
- Optional experiment/sample files can be loaded in batches; one selected sample can be shown as `sample`, `sample - observation`, or `sample - baseline`.
- Heatmap modes include absolute values and overall-mean anomalies, following the style of the existing E3SM ARM97 heatmap report.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import timedelta
import html
import io
import os
from pathlib import Path
import re


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "e3sm_scm_run_scripts_baseline").exists():
            return candidate
    return Path("/Users/yunlong/Workshop/SCM-UQ-Workflow")


ROOT = Path(os.environ.get("SCM_UQ_WORKFLOW_ROOT", find_repo_root())).resolve()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".local_cache/matplotlib-cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".local_cache"))

import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("repo root:", ROOT)


## Configuration

The defaults point to the current ARM97 baseline, ARM97 observation file, and the 70-member qmc14x5 stitched experiment. Set `INCLUDE_EXPERIMENT_MEMBERS = True` to enable sample heatmaps.


In [ ]:
DEFAULT_BASELINE = (
    ROOT
    / "e3sm_scm_run_scripts_baseline/baseline-output/scm_ARM97_baseline/run"
    / "case_scripts.eam.h0.1997-06-19-84585.nc"
)
DEFAULT_OBSERVATION = ROOT / "e3sm_scm_run_scripts_baseline/ARM97_iopfile_4scam.nc"
DEFAULT_EXPERIMENT_DIR = (
    ROOT
    / "arm97_experiments_0602/arm97_qmc14x5_stitched_seed20260602/mac/stitched"
)

BASELINE = Path(os.environ.get("SCM_BASELINE_HISTORY_FILE", DEFAULT_BASELINE)).expanduser().resolve()
OBSERVATION = Path(os.environ.get("ARM97_IOP_FILE", DEFAULT_OBSERVATION)).expanduser().resolve()
OUT_DIR = Path(os.environ.get("ARM97_PROFILE_HEATMAP_OUT_DIR", ROOT / "baseline_arm97_comparison/profile_heatmaps")).expanduser().resolve()

MAX_PRESSURE_PA = 96500.0
DEFAULT_VARIABLE = "T"
INCLUDE_LEVEL_VARIABLES: list[str] | None = None
EXCLUDE_LEVEL_VARIABLES = set()

INCLUDE_EXPERIMENT_MEMBERS = True
EXPERIMENT_DIR = Path(os.environ.get("SCM_EXPERIMENT_HISTORY_DIR", DEFAULT_EXPERIMENT_DIR)).expanduser().resolve()
EXPERIMENT_FILE_GLOB = os.environ.get("SCM_EXPERIMENT_FILE_GLOB", "mac_ARM97_qmc14x5_*_stitched_26day.nc")
EXPERIMENT_RUN_ID_REGEX = os.environ.get("SCM_EXPERIMENT_RUN_ID_REGEX", r"_(\d{3})_stitched_26day\.nc$")
BATCH_SIZE = 10
BATCH_INDEX = 0
SELECTED_RUN_IDS: list[int | str] | None = None
DEFAULT_SAMPLE_RUN_ID: int | str | None = None

DIVERGING_CMAP = "RdBu_r"
ABSOLUTE_CMAP = "viridis"
ROBUST_PERCENTILE = 99

assert BASELINE.exists(), BASELINE
assert OBSERVATION.exists(), OBSERVATION
if INCLUDE_EXPERIMENT_MEMBERS:
    assert EXPERIMENT_DIR.exists(), EXPERIMENT_DIR

print("baseline:", BASELINE)
print("observation:", OBSERVATION)
print("figure output:", OUT_DIR)
print("include experiment members:", INCLUDE_EXPERIMENT_MEMBERS)


## Observation Mapping

Only clear ARM97 pressure-level correspondences are mapped by default. Variables not listed here remain baseline-only unless you add a verified mapping.


In [ ]:
@dataclass(frozen=True)
class LevelSpec:
    model: str
    obs: str | None
    units: str
    description: str = ""
    scale_obs: float = 1.0
    obs_offset: float = 0.0


OBSERVED_LEVEL_SPECS = {
    "T": LevelSpec("T", "T", "K", "temperature"),
    "Q": LevelSpec("Q", "q", "kg/kg", "specific humidity"),
    "U": LevelSpec("U", "u", "m/s", "zonal wind"),
    "V": LevelSpec("V", "v", "m/s", "meridional wind"),
    "OMEGA": LevelSpec("OMEGA", "omega", "Pa/s", "pressure vertical velocity"),
    "RELHUM": LevelSpec("RELHUM", "rh", "%", "relative humidity"),
}


## Helpers


In [ ]:
def filled(arr):
    return np.asarray(np.ma.asarray(arr, dtype=np.float64).filled(np.nan), dtype=np.float64)


def as_time_series(var):
    data = np.ma.asarray(var[:], dtype=np.float64)
    if data.ndim == 1:
        return filled(data)
    axes = tuple(range(1, data.ndim))
    return filled(np.ma.mean(data, axis=axes))


def load_time_axis(ds):
    t = ds.variables["time"]
    days = np.asarray(t[:], dtype=np.float64)
    dates = np.array(num2date(days, t.units, getattr(t, "calendar", "standard"), only_use_cftime_datetimes=False))
    return days, dates


def interpolate_time_series(source_days, source_values, target_days):
    source_values = np.asarray(source_values, dtype=np.float64)
    finite = np.isfinite(source_values)
    if finite.sum() < 2:
        return np.full_like(target_days, np.nan, dtype=np.float64)
    return np.interp(target_days, source_days[finite], source_values[finite], left=np.nan, right=np.nan)


def interpolate_time_level(source_days, source_matrix, target_days):
    source_matrix = np.asarray(source_matrix, dtype=np.float64)
    out = np.full((len(target_days), source_matrix.shape[1]), np.nan, dtype=np.float64)
    for j in range(source_matrix.shape[1]):
        out[:, j] = interpolate_time_series(source_days, source_matrix[:, j], target_days)
    return out


def is_numeric_time_level_var(var) -> bool:
    dims = getattr(var, "dimensions", ())
    if not dims or dims[0] != "time":
        return False
    if "lev" not in dims and "ilev" not in dims:
        return False
    try:
        return np.issubdtype(np.dtype(var.dtype), np.number)
    except TypeError:
        return False


def vertical_dim_for(var) -> str:
    dims = getattr(var, "dimensions", ())
    if "lev" in dims:
        return "lev"
    if "ilev" in dims:
        return "ilev"
    raise ValueError(f"No lev/ilev dimension in {getattr(var, 'name', var)}")


def time_level_matrix(var, vertical_dim: str) -> np.ndarray:
    dims = list(var.dimensions)
    data = np.ma.asarray(var[:], dtype=np.float64)
    time_axis = dims.index("time")
    level_axis = dims.index(vertical_dim)
    data = np.moveaxis(data, [time_axis, level_axis], [0, 1])
    if data.ndim > 2:
        data = np.ma.mean(data, axis=tuple(range(2, data.ndim)))
    return filled(data)


def model_pressure_matrix(ds, vertical_dim: str) -> np.ndarray:
    p0 = float(np.asarray(ds.variables["P0"][...]))
    if vertical_dim == "lev":
        hya = np.asarray(ds.variables["hyam"][:], dtype=np.float64)
        hyb = np.asarray(ds.variables["hybm"][:], dtype=np.float64)
    elif vertical_dim == "ilev":
        hya = np.asarray(ds.variables["hyai"][:], dtype=np.float64)
        hyb = np.asarray(ds.variables["hybi"][:], dtype=np.float64)
    else:
        raise ValueError(vertical_dim)
    ps = as_time_series(ds.variables["PS"])
    return hya[None, :] * p0 + hyb[None, :] * ps[:, None]


def interp_matrix_to_pressures(values: np.ndarray, pressure: np.ndarray, target_pressures: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    pressure = np.asarray(pressure, dtype=np.float64)
    target_pressures = np.asarray(target_pressures, dtype=np.float64)
    out = np.full((values.shape[0], target_pressures.size), np.nan, dtype=np.float64)
    for i in range(values.shape[0]):
        p = pressure[i]
        v = values[i]
        finite = np.isfinite(p) & np.isfinite(v)
        if finite.sum() < 2:
            continue
        p = p[finite]
        v = v[finite]
        order = np.argsort(p)
        p = p[order]
        v = v[order]
        unique_p, unique_idx = np.unique(p, return_index=True)
        if unique_p.size < 2:
            continue
        in_range = (target_pressures >= unique_p[0]) & (target_pressures <= unique_p[-1])
        out[i, in_range] = np.interp(target_pressures[in_range], unique_p, v[unique_idx])
    return out


def obs_expression_available(obs, expression: str | None) -> bool:
    return expression is not None and expression in obs.variables


def obs_time_level_matrix(var) -> np.ndarray:
    dims = list(var.dimensions)
    data = np.ma.asarray(var[:], dtype=np.float64)
    time_axis = dims.index("time")
    level_axis = dims.index("lev")
    data = np.moveaxis(data, [time_axis, level_axis], [0, 1])
    if data.ndim > 2:
        data = np.ma.mean(data, axis=tuple(range(2, data.ndim)))
    return filled(data)


def parse_run_id(path: Path, regex: str = EXPERIMENT_RUN_ID_REGEX):
    match = re.search(regex, path.name)
    if not match:
        return None
    value = match.group(1)
    try:
        return int(value)
    except ValueError:
        return value


def sort_key(value):
    if isinstance(value, int):
        return (0, value)
    return (1, str(value))


def run_label(run_id) -> str:
    return f"{run_id:03d}" if isinstance(run_id, int) else str(run_id)


def discover_experiment_files() -> pd.DataFrame:
    rows = []
    for index, path in enumerate(sorted(EXPERIMENT_DIR.glob(EXPERIMENT_FILE_GLOB))):
        run_id = parse_run_id(path)
        rows.append({"run_id": index if run_id is None else run_id, "path": path})
    if not rows:
        raise FileNotFoundError(f"No files matched {EXPERIMENT_FILE_GLOB!r} in {EXPERIMENT_DIR}")
    return pd.DataFrame(rows).sort_values("run_id", key=lambda s: s.map(sort_key)).reset_index(drop=True)


def select_batch(files: pd.DataFrame, batch_size: int, batch_index: int, selected_run_ids=None) -> pd.DataFrame:
    if selected_run_ids is not None:
        selected = files[files["run_id"].isin(selected_run_ids)].copy()
        missing = sorted(set(selected_run_ids) - set(selected["run_id"]), key=sort_key)
        if missing:
            raise ValueError(f"Missing requested run ids: {missing}")
        return selected.sort_values("run_id", key=lambda s: s.map(sort_key)).reset_index(drop=True)
    if batch_size <= 0:
        raise ValueError("BATCH_SIZE must be positive")
    start = batch_index * batch_size
    stop = start + batch_size
    selected = files.iloc[start:stop].copy()
    if selected.empty:
        n_batches = int(np.ceil(len(files) / batch_size))
        raise ValueError(f"BATCH_INDEX {batch_index} is empty; valid range is 0..{n_batches - 1}")
    return selected.reset_index(drop=True)


## Discover Variables And Load Baseline/Observation


In [ ]:
def discover_level_specs() -> list[LevelSpec]:
    include = set(INCLUDE_LEVEL_VARIABLES) if INCLUDE_LEVEL_VARIABLES is not None else None
    specs = []
    with Dataset(BASELINE) as baseline, Dataset(OBSERVATION) as obs:
        for name, var in baseline.variables.items():
            if name in EXCLUDE_LEVEL_VARIABLES:
                continue
            if include is not None and name not in include:
                continue
            if not is_numeric_time_level_var(var):
                continue
            if name in OBSERVED_LEVEL_SPECS and obs_expression_available(obs, OBSERVED_LEVEL_SPECS[name].obs):
                specs.append(OBSERVED_LEVEL_SPECS[name])
            else:
                units = getattr(var, "units", "") or ""
                description = getattr(var, "long_name", "") or name
                specs.append(LevelSpec(name, None, units, description))
    if not specs:
        raise ValueError("No level variables discovered")
    return specs


def load_baseline_observation_heatmap_data(specs=None):
    specs = discover_level_specs() if specs is None else list(specs)
    data = {}
    rows = []
    with Dataset(BASELINE) as baseline, Dataset(OBSERVATION) as obs:
        baseline_days, baseline_dates = load_time_axis(baseline)
        obs_days = (np.asarray(obs.variables["tsec"][:], dtype=np.float64) - float(obs.variables["tsec"][0])) / 86400.0
        origin = baseline_dates[0] - timedelta(days=float(baseline_days[0]))
        obs_dates = np.array([origin + timedelta(days=float(x)) for x in obs_days])
        obs_levels_pa_all = np.asarray(obs.variables["lev"][:], dtype=np.float64)
        level_mask = obs_levels_pa_all <= MAX_PRESSURE_PA
        target_pressures_pa = obs_levels_pa_all[level_mask]

        pressure_cache = {}
        for spec in specs:
            if spec.model not in baseline.variables:
                continue
            var = baseline.variables[spec.model]
            vertical_dim = vertical_dim_for(var)
            if vertical_dim not in pressure_cache:
                pressure_cache[vertical_dim] = model_pressure_matrix(baseline, vertical_dim)
            baseline_matrix = time_level_matrix(var, vertical_dim)
            baseline_pressure = pressure_cache[vertical_dim]
            baseline_on_pressure = interp_matrix_to_pressures(baseline_matrix, baseline_pressure, target_pressures_pa)

            has_obs = obs_expression_available(obs, spec.obs)
            obs_native = None
            obs_at_baseline_time = None
            baseline_minus_obs = None
            if has_obs:
                obs_matrix = obs_time_level_matrix(obs.variables[spec.obs]) * spec.scale_obs + spec.obs_offset
                obs_native = obs_matrix[:, level_mask]
                obs_at_baseline_time = interpolate_time_level(obs_days, obs_native, baseline_days)
                baseline_minus_obs = baseline_on_pressure - obs_at_baseline_time

            data[spec.model] = {
                "spec": spec,
                "has_obs": has_obs,
                "vertical_dim": vertical_dim,
                "baseline_days": baseline_days,
                "baseline_dates": baseline_dates,
                "obs_days": obs_days,
                "obs_dates": obs_dates,
                "pressure_pa": target_pressures_pa,
                "baseline": baseline_on_pressure,
                "observation": obs_native,
                "obs_at_baseline_time": obs_at_baseline_time,
                "baseline_minus_obs": baseline_minus_obs,
            }
            rows.append({
                "variable": spec.model,
                "observation": spec.obs if has_obs else "",
                "description": spec.description,
                "units": spec.units,
                "vertical_dim": vertical_dim,
                "has_observation": has_obs,
                "n_pressure_levels": int(target_pressures_pa.size),
            })
    return data, pd.DataFrame(rows)


LEVEL_SPECS = discover_level_specs()
HEATMAP_DATA, VARIABLE_SUMMARY = load_baseline_observation_heatmap_data(LEVEL_SPECS)
with_obs = sum(item["has_obs"] for item in HEATMAP_DATA.values())
print(f"discovered {len(LEVEL_SPECS)} level variables")
print(f"variables with observation: {with_obs}; baseline-only variables: {len(HEATMAP_DATA) - with_obs}")
VARIABLE_SUMMARY.head(20)


## Optional Experiment Samples

If sample loading is enabled, this cell discovers the selected batch and loads one default sample for heatmap comparison. In the interactive controls you can switch among samples in the batch.


In [ ]:
def selected_experiment_batch():
    if not INCLUDE_EXPERIMENT_MEMBERS:
        return None, None
    all_files = discover_experiment_files()
    batch_files = select_batch(all_files, BATCH_SIZE, BATCH_INDEX, SELECTED_RUN_IDS)
    return all_files, batch_files


def load_sample_for_all_variables(sample_path: Path, specs=None):
    specs = LEVEL_SPECS if specs is None else list(specs)
    sample_data = {}
    with Dataset(sample_path) as sample:
        sample_days, sample_dates = load_time_axis(sample)
        pressure_cache = {}
        for spec in specs:
            if spec.model not in sample.variables or spec.model not in HEATMAP_DATA:
                continue
            var = sample.variables[spec.model]
            if not is_numeric_time_level_var(var):
                continue
            vertical_dim = vertical_dim_for(var)
            if vertical_dim not in pressure_cache:
                pressure_cache[vertical_dim] = model_pressure_matrix(sample, vertical_dim)
            sample_matrix = time_level_matrix(var, vertical_dim)
            sample_pressure = pressure_cache[vertical_dim]
            pressure_pa = HEATMAP_DATA[spec.model]["pressure_pa"]
            sample_on_pressure = interp_matrix_to_pressures(sample_matrix, sample_pressure, pressure_pa)
            baseline_at_sample_time = interpolate_time_level(
                HEATMAP_DATA[spec.model]["baseline_days"],
                HEATMAP_DATA[spec.model]["baseline"],
                sample_days,
            )
            payload = {
                "sample_days": sample_days,
                "sample_dates": sample_dates,
                "sample": sample_on_pressure,
                "sample_minus_baseline": sample_on_pressure - baseline_at_sample_time,
            }
            if HEATMAP_DATA[spec.model]["has_obs"]:
                obs_at_sample_time = interpolate_time_level(
                    HEATMAP_DATA[spec.model]["obs_days"],
                    HEATMAP_DATA[spec.model]["observation"],
                    sample_days,
                )
                payload["sample_minus_obs"] = sample_on_pressure - obs_at_sample_time
            sample_data[spec.model] = payload
    return sample_data


ALL_EXPERIMENT_FILES, BATCH_FILES = selected_experiment_batch()
SAMPLE_CACHE = {}
DEFAULT_SAMPLE_ID = None
if BATCH_FILES is not None:
    if DEFAULT_SAMPLE_RUN_ID is None:
        DEFAULT_SAMPLE_ID = BATCH_FILES.iloc[0]["run_id"]
    else:
        DEFAULT_SAMPLE_ID = DEFAULT_SAMPLE_RUN_ID
    default_row = BATCH_FILES[BATCH_FILES["run_id"] == DEFAULT_SAMPLE_ID]
    if default_row.empty:
        raise ValueError(f"DEFAULT_SAMPLE_RUN_ID {DEFAULT_SAMPLE_ID!r} is not in selected batch")
    print(f"experiment files: {len(ALL_EXPERIMENT_FILES)}; selected batch: {[run_label(x) for x in BATCH_FILES['run_id']]}")
    print(f"loading default sample {run_label(DEFAULT_SAMPLE_ID)}")
    SAMPLE_CACHE[DEFAULT_SAMPLE_ID] = load_sample_for_all_variables(Path(default_row.iloc[0]["path"]))
else:
    print("experiment sample heatmaps are disabled")


## Heatmap Plotting Helpers


In [ ]:
def default_variable():
    for name in [DEFAULT_VARIABLE, "Q", "RELHUM", "OMEGA", "CLOUD"]:
        if name in HEATMAP_DATA:
            return name
    return next(iter(HEATMAP_DATA))


def robust_abs_limit(values, percentile=ROBUST_PERCENTILE):
    values = np.asarray(values, dtype=np.float64)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return 1.0
    limit = np.nanpercentile(np.abs(finite), percentile)
    if not np.isfinite(limit) or limit == 0:
        limit = np.nanmax(np.abs(finite))
    if not np.isfinite(limit) or limit == 0:
        limit = 1.0
    return float(limit)


def finite_minmax(values):
    values = np.asarray(values, dtype=np.float64)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return 0.0, 1.0
    vmin = float(np.nanmin(finite))
    vmax = float(np.nanmax(finite))
    if vmin == vmax:
        pad = abs(vmin) * 0.01 if vmin != 0 else 1.0
        return vmin - pad, vmax + pad
    return vmin, vmax


def get_sample_payload(sample_id):
    if sample_id is None:
        return None
    if sample_id not in SAMPLE_CACHE:
        row = BATCH_FILES[BATCH_FILES["run_id"] == sample_id]
        if row.empty:
            raise ValueError(f"sample {sample_id!r} is not in selected batch")
        SAMPLE_CACHE[sample_id] = load_sample_for_all_variables(Path(row.iloc[0]["path"]))
    return SAMPLE_CACHE[sample_id]


def field_for_plot(var: str, field: str, mode: str = "absolute", sample_id=None):
    item = HEATMAP_DATA[var]
    sample_payload = get_sample_payload(sample_id)

    if field == "baseline":
        dates = item["baseline_dates"]
        values = item["baseline"]
    elif field == "observation" and item["has_obs"]:
        dates = item["obs_dates"]
        values = item["observation"]
    elif field == "baseline_minus_obs" and item["has_obs"]:
        dates = item["baseline_dates"]
        values = item["baseline_minus_obs"]
    elif field == "sample" and sample_payload is not None and var in sample_payload:
        dates = sample_payload[var]["sample_dates"]
        values = sample_payload[var]["sample"]
    elif field == "sample_minus_obs" and item["has_obs"] and sample_payload is not None and var in sample_payload:
        dates = sample_payload[var]["sample_dates"]
        values = sample_payload[var]["sample_minus_obs"]
    elif field == "sample_minus_baseline" and sample_payload is not None and var in sample_payload:
        dates = sample_payload[var]["sample_dates"]
        values = sample_payload[var]["sample_minus_baseline"]
    else:
        raise ValueError(f"field {field!r} is unavailable for {var}")

    values = np.asarray(values, dtype=np.float64)
    if mode == "overall_anomaly":
        values = values - np.nanmean(values)
    return dates, item["pressure_pa"], values


def field_options_for(var: str):
    item = HEATMAP_DATA[var]
    options = [("Baseline", "baseline")]
    if item["has_obs"]:
        options = [("Observation", "observation"), ("Baseline", "baseline"), ("Baseline - observation", "baseline_minus_obs")]
    if INCLUDE_EXPERIMENT_MEMBERS and BATCH_FILES is not None:
        options.append(("Sample", "sample"))
        if item["has_obs"]:
            options.append(("Sample - observation", "sample_minus_obs"))
        options.append(("Sample - baseline", "sample_minus_baseline"))
    return options


def plot_heatmap(var=None, field="baseline", mode="absolute", sample_id=None, ax=None):
    var = var or default_variable()
    dates, pressure_pa, values = field_for_plot(var, field, mode=mode, sample_id=sample_id)
    spec = HEATMAP_DATA[var]["spec"]
    units = spec.units or var
    long_name = spec.description or var

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5.8))
    else:
        fig = ax.figure

    if mode == "overall_anomaly" or "minus" in field:
        limit = robust_abs_limit(values)
        norm = TwoSlopeNorm(vcenter=0.0, vmin=-limit, vmax=limit)
        cmap = DIVERGING_CMAP
    else:
        vmin, vmax = finite_minmax(values)
        norm = None
        cmap = ABSOLUTE_CMAP

    mesh = ax.pcolormesh(dates, pressure_pa / 100.0, values.T, shading="auto", cmap=cmap, norm=norm, vmin=None if norm else vmin, vmax=None if norm else vmax)
    ax.invert_yaxis()
    ax.set_xlabel("Time")
    ax.set_ylabel("Pressure (hPa)")
    field_label = dict(field_options_for(var)).get(field, field)
    mode_label = "overall mean anomaly" if mode == "overall_anomaly" else "absolute value"
    sample_label = f" | sample {run_label(sample_id)}" if sample_id is not None and field.startswith("sample") else ""
    ax.set_title(f"{var}: {field_label}{sample_label}\n{long_name} | {mode_label}")
    cbar = fig.colorbar(mesh, ax=ax, pad=0.015)
    cbar.set_label(units)
    fig.autofmt_xdate(rotation=0)
    return fig, ax


def scale_note(var, field="baseline", mode="absolute", sample_id=None):
    _, _, values = field_for_plot(var, field, mode=mode, sample_id=sample_id)
    spec = HEATMAP_DATA[var]["spec"]
    units = spec.units or var
    if mode == "overall_anomaly" or "minus" in field:
        limit = robust_abs_limit(values)
        return f"Color scale: {DIVERGING_CMAP}, zero-centered at [{-limit:.4g}, {limit:.4g}] ({units})."
    vmin, vmax = finite_minmax(values)
    return f"Color scale: {ABSOLUTE_CMAP}, autoscaled at [{vmin:.4g}, {vmax:.4g}] ({units})."


## Static Preview


In [ ]:
preview_var = default_variable()
preview_field = "baseline_minus_obs" if HEATMAP_DATA[preview_var]["has_obs"] else "baseline"
print(scale_note(preview_var, preview_field, mode="absolute", sample_id=DEFAULT_SAMPLE_ID))
fig, ax = plot_heatmap(preview_var, preview_field, mode="absolute", sample_id=DEFAULT_SAMPLE_ID)
plt.show()


## Interactive Heatmap

Changing the variable refreshes the available field choices. Sample choices are shown only when `INCLUDE_EXPERIMENT_MEMBERS = True`.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    var_dropdown = widgets.Dropdown(
        options=list(HEATMAP_DATA),
        value=default_variable(),
        description="Variable",
        layout=widgets.Layout(width="460px"),
    )
    field_dropdown = widgets.Dropdown(
        options=field_options_for(var_dropdown.value),
        value=field_options_for(var_dropdown.value)[0][1],
        description="Field",
        layout=widgets.Layout(width="460px"),
    )
    mode_dropdown = widgets.Dropdown(
        options=[("Absolute value", "absolute"), ("Overall mean anomaly", "overall_anomaly")],
        value="absolute",
        description="Mode",
        layout=widgets.Layout(width="460px"),
    )
    if INCLUDE_EXPERIMENT_MEMBERS and BATCH_FILES is not None:
        sample_dropdown = widgets.Dropdown(
            options=[(run_label(x), x) for x in BATCH_FILES["run_id"]],
            value=DEFAULT_SAMPLE_ID,
            description="Sample",
            layout=widgets.Layout(width="460px"),
        )
    else:
        sample_dropdown = widgets.Dropdown(options=[("none", None)], value=None, description="Sample", layout=widgets.Layout(width="460px"), disabled=True)

    scale_widget = widgets.HTML()
    image_widget = widgets.Image(format="png", layout=widgets.Layout(width="100%"))

    def render_heatmap_png(var, field, mode, sample_id):
        fig, ax = plot_heatmap(var, field, mode=mode, sample_id=sample_id)
        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", dpi=120, bbox_inches="tight")
        plt.close(fig)
        return buffer.getvalue()

    def update_field_options(*_):
        options = field_options_for(var_dropdown.value)
        old = field_dropdown.value
        field_dropdown.options = options
        field_dropdown.value = old if old in [value for _, value in options] else options[0][1]

    def update_heatmap(*_):
        sample_id = sample_dropdown.value if INCLUDE_EXPERIMENT_MEMBERS else None
        scale_widget.value = f"<div style='font-size: 14px; margin: 8px 0;'>{html.escape(scale_note(var_dropdown.value, field_dropdown.value, mode_dropdown.value, sample_id))}</div>"
        image_widget.value = render_heatmap_png(var_dropdown.value, field_dropdown.value, mode_dropdown.value, sample_id)

    var_dropdown.observe(update_field_options, names="value")
    for control in [var_dropdown, field_dropdown, mode_dropdown, sample_dropdown]:
        control.observe(update_heatmap, names="value")

    update_field_options()
    update_heatmap()
    display(widgets.VBox([var_dropdown, field_dropdown, mode_dropdown, sample_dropdown, scale_widget, image_widget]))
except Exception as exc:
    print("ipywidgets heatmap controls are unavailable in this kernel.")
    print(repr(exc))


## Static Export

This exports a compact set of default heatmaps for each variable: `baseline`, `observation` and `baseline - observation` when available, plus selected sample fields when samples are enabled.


In [ ]:
def safe_name(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text)


def default_export_fields(var: str):
    fields = ["baseline"]
    if HEATMAP_DATA[var]["has_obs"]:
        fields = ["observation", "baseline", "baseline_minus_obs"]
    if INCLUDE_EXPERIMENT_MEMBERS and DEFAULT_SAMPLE_ID is not None:
        fields.append("sample")
        if HEATMAP_DATA[var]["has_obs"]:
            fields.append("sample_minus_obs")
        fields.append("sample_minus_baseline")
    return fields


def export_heatmaps(vars_to_export=None, mode="absolute", sample_id=DEFAULT_SAMPLE_ID, out_dir: Path = OUT_DIR):
    vars_to_export = list(HEATMAP_DATA) if vars_to_export is None else list(vars_to_export)
    sample_label = "no_samples" if sample_id is None else f"sample_{run_label(sample_id)}"
    fig_dir = out_dir / mode / sample_label
    fig_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for var in vars_to_export:
        for field in default_export_fields(var):
            try:
                fig, ax = plot_heatmap(var, field, mode=mode, sample_id=sample_id)
            except ValueError:
                continue
            path = fig_dir / f"{safe_name(var)}_{safe_name(field)}_{mode}.png"
            fig.savefig(path, dpi=160, bbox_inches="tight")
            plt.close(fig)
            paths.append(path)
    return paths


OUT_DIR.mkdir(parents=True, exist_ok=True)
summary_out = OUT_DIR / "pressure_level_heatmap_variable_summary.csv"
VARIABLE_SUMMARY.to_csv(summary_out, index=False)
# Keep the default export modest. Set vars_to_export=None to export every discovered variable.
paths = export_heatmaps(vars_to_export=[default_variable()], mode="absolute", sample_id=DEFAULT_SAMPLE_ID)
print(summary_out)
print(f"exported {len(paths)} preview heatmaps to {paths[0].parent if paths else OUT_DIR}")
paths


## Notes

- `baseline`, `sample`, and `observation` are shown as absolute values by default with `viridis`.
- Difference fields and anomaly mode use `RdBu_r`, centered at zero.
- For sample heatmaps, the notebook loads only the selected sample on demand and caches it in memory.
